### Why ML models overfit data 

In [44]:
#evalauting model based on same datatset as used to train = poor results 
#the model is not able to geenralise on unseen data 
#thf to eval investment stratgy, use datat outsid eof the data used to train model

import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', parse_dates=['Date'], index_col=0)
df = df.drop(columns='change_tomorrow_direction')
df

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2016-12-08,55.079998,55.594599,54.926523,55.341812,21220800,1.549152
2016-12-09,55.946697,55.964754,55.188343,55.233482,27349400,0.321693
2016-12-12,56.127254,56.244620,55.720996,55.811275,20198100,1.286142
2016-12-13,56.858536,57.255768,56.190463,56.425191,35718900,-0.478626
2016-12-14,56.587692,57.282851,56.452270,56.876589,30352700,-0.159799
...,...,...,...,...,...,...
2025-05-16,454.269989,454.359985,448.730011,452.049988,23849800,1.002464
2025-05-19,458.869995,459.589996,450.799988,450.880005,21336500,-0.152778
2025-05-20,458.170013,458.339996,454.320007,455.589996,15441800,-1.237379


In [2]:
target = df.change_tomorrow 
explanatory = df[['Close', 'High', 'Low', 'Open','Volume']]

In [3]:
#train-test split 
n_days = len(df.index)
n_days

2125

In [4]:
n_days_split = int(n_days*0.70)
n_days_split

1487

In [5]:
X_train, y_train = explanatory.iloc[:n_days_split], target.iloc[:n_days_split]
X_test, y_test = explanatory.iloc[n_days_split:], target.iloc[n_days_split:]

In [6]:
X_train

,Close,High,Low,Open,Volume
Date,,,,,
2016-12-08,55.079998,55.594599,54.926523,55.341812,21220800
2016-12-09,55.946697,55.964754,55.188343,55.233482,27349400
2016-12-12,56.127254,56.244620,55.720996,55.811275,20198100
2016-12-13,56.858536,57.255768,56.190463,56.425191,35718900
2016-12-14,56.587692,57.282851,56.452270,56.876589,30352700
...,...,...,...,...,...
2022-10-28,230.523178,231.236641,220.925790,221.111486,40647700
2022-10-31,226.867966,229.594714,225.910170,228.461006,28357300
2022-11-01,222.997711,230.396117,222.176756,229.281960,30592300


In [7]:
X_test

,Close,High,Low,Open,Volume
Date,,,,,
2022-11-04,216.371414,216.566877,208.591849,212.618465,36789100
2022-11-07,222.704529,223.232296,216.263918,216.957830,33498000
2022-11-08,223.681854,226.398835,220.720541,223.515710,28192500
2022-11-09,219.420685,223.447301,219.244772,222.215853,27852900
2022-11-10,237.472000,237.814072,229.672899,230.093144,46268000
...,...,...,...,...,...
2025-05-16,454.269989,454.359985,448.730011,452.049988,23849800
2025-05-19,458.869995,459.589996,450.799988,450.880005,21336500
2025-05-20,458.170013,458.339996,454.320007,455.589996,15441800


In [8]:
#fit model on train set 

from sklearn.tree import DecisionTreeRegressor 

model_dtr_split = DecisionTreeRegressor(max_depth=15, random_state=42)
model_dtr_split.fit(X=X_train, y=y_train)

DecisionTreeRegressor(max_depth=15, random_state=42)

In [9]:
y_pred_test = model_dtr_split.predict(X=X_test)

In [10]:
#eval model on test set 
from sklearn.metrics import mean_squared_error 

mean_squared_error(y_true= y_test, y_pred=y_pred_test)

4.589258212249226

In [11]:
#eval on test set - show overfitting 
y_pred_train = model_dtr_split.predict(X=X_train)
mean_squared_error(y_true= y_train, y_pred=y_pred_train)
#lower the mse = better performing the mdoel 
#closer mse to std = better performing model 

1.20063103103932

### Train model within the backtest 

In [12]:
from backtesting import Backtest, Strategy

c:\Users\thelm\AppData\Local\Programs\Python\Python313\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [13]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5

    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False

        self.model.fit(X=X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :] 
        forecast_tomorrow = self.model.predict(explanatory_today)[0] 

        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False: #5% inc 
            self.buy() 
            self.already_bought = True 
        elif forecast_tomorrow < self.limit_sell and self.already_bought== True: #5% dec
            self.sell() 
            self.already_bought = False
        else: 
            pass
            

In [14]:
#run backtest on test data 

bt_test = Backtest(X_test, Regression, cash=10000, commission=0.002, exclusive_orders=True)

In [15]:
results_test = bt_test.run(limit_buy=1, limit_sell=-5)

In [16]:
df_results_test = results_test.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values': 'Out of Sample (Test)'}, axis=1)
df_results_test

,Out of Sample (Test)
Start,2022-11-04 00:00:00
End,2025-05-22 00:00:00
Duration,930 days 00:00:00
Exposure Time [%],0.0
Equity Final [$],18381.010234
Equity Peak [$],18746.710918
Return [%],83.810102


In [17]:
#run backtest on train data 

bt_train = Backtest(X_train, Regression, cash=10000, commission=0.002, exclusive_orders=True)

In [18]:
results_train = bt_train.run(limit_buy=1, limit_sell=-5)
df_results_train = results_train.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values': 'In Sample (Train)'}, axis=1)
df_results_train

,In Sample (Train)
Start,2016-12-08 00:00:00
End,2022-11-03 00:00:00
Duration,2156 days 00:00:00
Exposure Time [%],91.190316
Equity Final [$],50048.853486
Equity Peak [$],62436.893488
Commissions [$],3019.033588
Return [%],400.488535


In [19]:
#compare both backtests 
df_results_comparison = pd.concat([df_results_test, df_results_train], axis=1)
df_results_comparison

,Out of Sample (Test),In Sample (Train)
Start,2022-11-04 00:00:00,2016-12-08 00:00:00
End,2025-05-22 00:00:00,2022-11-03 00:00:00
Duration,930 days 00:00:00,2156 days 00:00:00
Exposure Time [%],0.0,91.190316
Equity Final [$],18381.010234,50048.853486
Equity Peak [$],18746.710918,62436.893488
Return [%],83.810102,400.488535
Commissions [$],NaN,3019.033588


### Anchored Walk forward within ML model

In [20]:
#more realistic - validates the data across diff time ranges 
from sklearn.model_selection import TimeSeriesSplit

ts = TimeSeriesSplit(test_size=200) #splitin 200 days
#advance 200 days for eevery split in the data

In [21]:
splits = ts.split(X=df)

In [22]:
split1 = next(splits)
split1

(array([   0,    1,    2, ..., 1122, 1123, 1124], shape=(1125,)),
 array([1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135,
        1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146,
        1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157,
        1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1166, 1167, 1168,
        1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1179,
        1180, 1181, 1182, 1183, 1184, 1185, 1186, 1187, 1188, 1189, 1190,
        1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199, 1200, 1201,
        1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1211, 1212,
        1213, 1214, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223,
        1224, 1225, 1226, 1227, 1228, 1229, 1230, 1231, 1232, 1233, 1234,
        1235, 1236, 1237, 1238, 1239, 1240, 1241, 1242, 1243, 1244, 1245,
        1246, 1247, 1248, 1249, 1250, 1251, 1252, 1253, 1254, 1255, 1256,
        1257, 1258, 1259, 1260, 1261, 1262, 12

In [23]:
split2 = next(splits)
split2

(array([   0,    1,    2, ..., 1322, 1323, 1324], shape=(1325,)),
 array([1325, 1326, 1327, 1328, 1329, 1330, 1331, 1332, 1333, 1334, 1335,
        1336, 1337, 1338, 1339, 1340, 1341, 1342, 1343, 1344, 1345, 1346,
        1347, 1348, 1349, 1350, 1351, 1352, 1353, 1354, 1355, 1356, 1357,
        1358, 1359, 1360, 1361, 1362, 1363, 1364, 1365, 1366, 1367, 1368,
        1369, 1370, 1371, 1372, 1373, 1374, 1375, 1376, 1377, 1378, 1379,
        1380, 1381, 1382, 1383, 1384, 1385, 1386, 1387, 1388, 1389, 1390,
        1391, 1392, 1393, 1394, 1395, 1396, 1397, 1398, 1399, 1400, 1401,
        1402, 1403, 1404, 1405, 1406, 1407, 1408, 1409, 1410, 1411, 1412,
        1413, 1414, 1415, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1423,
        1424, 1425, 1426, 1427, 1428, 1429, 1430, 1431, 1432, 1433, 1434,
        1435, 1436, 1437, 1438, 1439, 1440, 1441, 1442, 1443, 1444, 1445,
        1446, 1447, 1448, 1449, 1450, 1451, 1452, 1453, 1454, 1455, 1456,
        1457, 1458, 1459, 1460, 1461, 1462, 14

In [24]:
list_df_train = []
list_df_test = []

In [25]:
for index_train, index_test in ts.split(df):
    list_df_train.append(df.iloc[index_train]) #index_train = 1st element of array
    list_df_test.append(df.iloc[index_test]) #index_test = 2nd element of array

In [26]:
list_df_train[0]

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2016-12-08,55.079998,55.594599,54.926523,55.341812,21220800,1.549152
2016-12-09,55.946697,55.964754,55.188343,55.233482,27349400,0.321693
2016-12-12,56.127254,56.244620,55.720996,55.811275,20198100,1.286142
2016-12-13,56.858536,57.255768,56.190463,56.425191,35718900,-0.478626
2016-12-14,56.587692,57.282851,56.452270,56.876589,30352700,-0.159799
...,...,...,...,...,...,...
2021-05-24,242.595108,242.962710,239.431829,239.702690,21411500,0.373436
2021-05-25,243.504440,244.500822,242.633820,243.552811,17704300,-0.091447
2021-05-26,243.281967,244.684640,242.566114,243.223913,17771600,-0.874410


In [27]:
list_df_train[1]

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2016-12-08,55.079998,55.594599,54.926523,55.341812,21220800,1.549152
2016-12-09,55.946697,55.964754,55.188343,55.233482,27349400,0.321693
2016-12-12,56.127254,56.244620,55.720996,55.811275,20198100,1.286142
2016-12-13,56.858536,57.255768,56.190463,56.425191,35718900,-0.478626
2016-12-14,56.587692,57.282851,56.452270,56.876589,30352700,-0.159799
...,...,...,...,...,...,...
2022-03-09,280.709137,281.779438,273.197612,275.785783,35204500,-1.018945
2022-03-10,277.877716,278.860451,273.003000,275.377111,30628000,-1.970939
2022-03-11,272.506775,281.691852,271.884044,280.183692,27209300,-1.313128


In [28]:
list_df_test[0]

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2021-06-01,239.325470,243.088509,238.899843,243.030470,23213300,-0.040445
2021-06-02,239.228714,241.134419,237.816358,240.031627,19406700,-0.647096
2021-06-03,237.690628,238.300056,235.069069,237.216615,25307700,2.025592
2021-06-04,242.604813,243.436745,239.431865,239.673706,25281100,1.189853
2021-06-07,245.526215,245.797075,241.656766,241.821215,23079200,-0.490938
...,...,...,...,...,...,...
2022-03-09,280.709137,281.779438,273.197612,275.785783,35204500,-1.018945
2022-03-10,277.877716,278.860451,273.003000,275.377111,30628000,-1.970939
2022-03-11,272.506775,281.691852,271.884044,280.183692,27209300,-1.313128


In [29]:
list_df_test[1]

,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,
2022-03-16,286.440094,286.615226,275.552274,281.302650,37826300,0.281146
2022-03-17,287.247681,287.627133,281.555652,285.369807,30816600,1.734160
2022-03-18,292.316925,292.871539,284.824880,287.393572,43390600,-0.424516
2022-03-21,291.081238,292.034784,286.936269,290.818540,28351200,1.611522
2022-03-22,295.848907,296.763525,290.701755,291.703939,27599700,-1.525940
...,...,...,...,...,...,...
2022-12-22,233.446640,237.170969,229.212662,236.455496,28651700,0.226186
2022-12-23,233.975861,234.113072,229.281257,231.408041,21207000,-0.746944
2022-12-27,232.241150,234.171905,231.133648,233.946490,16688600,-1.036127


In [30]:
#add ml model within for loop 
y = df.change_tomorrow 
X = df[['Close','High', 'Low', 'Open', 'Volume']]

list_df_train = []
list_df_test = []

for index_train, index_test in ts.split(df):
    list_df_train.append(df.iloc[index_train]) 
    list_df_test.append(df.iloc[index_test]) 
    X_train, y_train = X.iloc[index_train], y.iloc[index_train]
    X_test, y_test = X.iloc[index_test], y.iloc[index_test]

In [31]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error


In [32]:
model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)
model_dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=15, random_state=42)

In [33]:
y_pred = model_dt.predict(X_test)

In [34]:
error_mse = mean_squared_error(y_test, y_pred)
error_mse

3.59788550961643

In [35]:
#add into for loop to get 5 diff errors
model_dt = DecisionTreeRegressor(max_depth=15, random_state=42)

error_mse_list = []

for index_train, index_test in ts.split(df): 
    X_train, y_train = X.iloc[index_train], y.iloc[index_train]
    X_test, y_test = X.iloc[index_test], y.iloc[index_test]

    model_dt.fit(X_train, y_train)

    y_pred = model_dt.predict(X_test)
    error_mse = mean_squared_error(y_test, y_pred)

    error_mse_list.append(error_mse)

In [36]:
error_mse_list

[12.046905785514207,
 5.688933809879191,
 4.68228641273671,
 7.698403614369335,
 3.59788550961643]

In [37]:
#eval overall error of ml mocel by computing avg 
import numpy as np

np.mean(error_mse_list) # this is in dollars 

np.float64(6.742883026423175)

### Anchored walk forward validation in backtesting 

In [38]:
#need to preprocess data within strategy class to implement 
#anchored alk forward in backtesting 
from backtesting import Backtest, Strategy

In [39]:
class AnchoredRegression(Strategy):
    limit_buy = 1
    limit_sell = -5
    n_train = 600
    coef_retrain = 200 

    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False

        #train model on 1st 600 days 
        X_train = self.data.df.iloc[:self.n_train, :-1] #everything but the last col
        y_train = self.data.df.iloc[:self.n_train, -1] #only the last col - chnage_tomorrow

        self.model.fit(X=X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :-1] #take everythigng but last col
        forecast_tomorrow = self.model.predict(explanatory_today)[0] 

        #improve stratgy to get higher equity via changign the numbers here or using the bt optimisation method
        if forecast_tomorrow > self.limit_buy and self.already_bought == False: 
            self.buy() 
            self.already_bought = True 
        elif forecast_tomorrow < self.limit_sell and self.already_bought== True: 
            self.sell() 
            self.already_bought = False
        else: 
            pass

#intialise the mdoel by first training it on the first 600 days 
            

In [46]:
# To take action on whether to buy/sell a stock req a new class that 
# Contains the anchored walk forward procedure 
# Once model has been intialised, starts the walk forward backtest 
# According to these conditions 

class WalkForwardAnchored(AnchoredRegression):
    def next(self):
        # Take no action and ove onto following day until thr data have been 
        # Trained for at least 600 days 
        if len(self.data) < self.n_train:
            return 
        
        # Retrain the model every 200 days after intial 600 day trianinig
        if len(self.data) % self.coef_retrain == 0:
            X_train = self.data.df.iloc[:, :-1]
            y_train = self.data.df.iloc[:, -1]

            self.model.fit(X_train, y_train)

            # Taking the next function from the inherited Regression clads
            super().next()
        
        else:
            super().next()

In [47]:
bt = Backtest(df, WalkForwardAnchored, cash=10000, commission=.002, exclusive_orders=True)

In [50]:
import multiprocessing as mp 
mp.set_start_method('spawn', force=True)

In [51]:
stats_skopt, heatmap, optimize_results = bt.optimize(
    limit_buy=range(0, 6),
    limit_sell=range(-6, 0),
    maximize='Return [%]',
    method='skopt',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True
)

C:\Users\thelm\AppData\Local\Temp\ipykernel_33032\3821890531.py:1: DeprecationWarning: `Backtest.optimize(method="skopt")` is deprecated. Use `method="sambo"`.
  stats_skopt, heatmap, optimize_results = bt.optimize(


In [52]:
dff = heatmap.reset_index()

In [53]:
dff = dff.sort_values('Return [%]', ascending=False)
dff

,limit_buy,limit_sell,Return [%]
15,3,-6,125.877482
9,2,-6,110.838114
0,0,-6,106.097171
3,1,-6,105.421773
21,4,-6,61.099001
23,4,-4,61.099001
22,4,-5,61.099001
26,5,-6,61.099001
27,5,-4,61.099001
16,3,-5,28.639754


### Create library for backtesting strategies

In [54]:
#create a a new python file - called it strategies .py
# Unachored walk forward - instea do f achoring a fixed date at the beginnning 
# Of the datatset  to train the model, the days in an achored approach moce 
# Along with the test set 
# Import the tartegies files as a library 

import strategies

strategies.WalkForwardUnAnchored

strategies.WalkForwardUnAnchored

In [55]:
bt_unanchored = Backtest(df, strategies.WalkForwardUnAnchored, cash=10000, commission=.002, exclusive_orders=True)

In [61]:
stats_skopt_unanchored, heatmap_unanchored, optimize_results_unanchored = bt_unanchored.optimize(
    limit_buy=range(0, 6),
    limit_sell=range(-6, 0),
    maximize='Return [%]',
    method='skopt',
    max_tries=500,
    random_state=42,
    return_heatmap=True,
    return_optimization=True
)

C:\Users\thelm\AppData\Local\Temp\ipykernel_33032\1143852671.py:1: DeprecationWarning: `Backtest.optimize(method="skopt")` is deprecated. Use `method="sambo"`.
  stats_skopt_unanchored, heatmap_unanchored, optimize_results_unanchored = bt_unanchored.optimize(


In [62]:
dff_unanchored = heatmap.reset_index()

In [63]:
dff_unanchored = dff_unanchored.sort_values('Return [%]', ascending=False)
dff_unanchored
#did wayyyyy better

,limit_buy,limit_sell,Return [%]
9,2,-6,134.958444
0,0,-6,122.924657
3,1,-6,114.678186
15,3,-6,102.291895
23,4,-4,51.334939
21,4,-6,51.334939
22,4,-5,51.334939
4,1,-5,28.587154
1,0,-4,10.134310
5,1,-4,0.177708


In [69]:
#look at reports to compare stratgies 
bt.plot(filename='anchored_walk_forward.html')
# Starts to take decisions on the invetsment very late - almost at the end of 2022 

GridPlot(id='p3441', ...)

In [70]:
bt_unanchored.plot(filename='UNanchored_walk_forward.html')
# Starts taking decisions from May 2019

GridPlot(id='p3795', ...)

### Interpret reports from walk forward validation appraoches 